# Урок 18: Осигуряване на AI агенти с криптографски разписки

## Практически тетрадки

Тази тетрадка разглежда четири упражнения:

1. **Подпишете първата си разписка** за обаждане към инструмент на агент и я проверете.
2. **Манипулирайте разписката** и наблюдавайте как проверката не успява.
3. **Създайте верига от три разписки** и потвърдете целостта на веригата.
4. **Опаковайте повикване към инструмент на Microsoft Agent Framework**, така че всяко действие да издава разписка.

Всички криптографски примитиви се импортират от добре поддържани библиотеки (`pynacl` за Ed25519, `jcs` за RFC 8785 каноничен JSON, `hashlib` от стандартната библиотека на Python за SHA-256). Самата логика за разписките е чист Python, който можете да прочетете и модифицирате.

Стартирайте клетките по ред. Всяка секция е кратка и самостоятелна.


## Настройка

Инсталирайте двете зависимости. И двете имат либерални лицензи (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Помощни инструменти

Тези два помощни инструмента обработват base64url кодирането (без допълващи символи) и SHA-256 хеширането на произволни обекти. Те позволяват останалата част от тетрадката да се съсредоточи върху логиката на разписката сама по себе си.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Раздел 1: Подпишете първото си разписване

Представете си, че наш агент за **Contoso Travel** току-що е намерил полети от Сидни до Лос Анджелис за клиент. Искаме да запишем този повикване на инструмента като подписано разписване, за да може бъдещ одитор да го провери без да ни се доверява.

### Стъпка 1.1: Генерирайте подписващ ключ

В продукция подписващият ключ на агента би се съхранявал в хардуерен модул за сигурност (HSM), Azure Key Vault или подобно защитено хранилище. За този урок генерираме нов ключ в паметта.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Стъпка 1.2: Създаване на полезния товар на разписката

Полезният товар съдържа всичко, което искаме разписката да засвидетелства: кой действа, с какъв инструмент, с какви аргументи, какъв е резултатът, по каква политика и кога. Хешираме аргументите и резултата, вместо да ги включваме директно, за да не се разкрива чувствително съдържание.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Стъпка 1.3: Подпишете и съберете разписката

Три стъпки:

1. Канонизирайте полезния товар, използвайки JCS, така че две имплементации, които произвеждат една и съща логическа разписка, да генерират идентични по байтове резултати.
2. Хеширайте канонизираните байтове с SHA-256.
3. Подпишете хеша с частния ключ Ed25519.

След това подписът се прилага към оригиналния полезен товар, за да се получи крайната разписка.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Стъпка 1.4: Проверете разписката

Проверката обръща процеса. Премахваме подписа, пресмятаме отново каноничния хеш и проверяваме подписа спрямо публичния ключ в разписката.

Одитор, правещ тази проверка, не се нуждае от нищо друго освен самата разписка. Няма нужда от повикване на услуга, заявяване към директория с ключове или изискване за доверие.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Трябва да видите `Receipt is valid: True`. Агентът е създал първия си криптографски подписан одиторски запис.


## Раздел 2: Манипулиране с касовата бележка

Целта на касовите бележки е да са защитени от манипулации. Нека го докажем.

Ще променим точно един символ от касовата бележка и ще наблюдаваме как проверката се проваля.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Какво току-що се случи?

Когато променихме `policy_id`, каноничните байтове се промениха. SHA-256 хешът на тези байтове се промени. Подписът (който беше върху първоначалния хеш) вече не съответства на новия хеш. Проверката правилно връща `False`.

Няма начин да се промени някое поле от разписката и тя да продължи да се проверява, освен ако нападателят няма частния ключ. Докато частният ключ е в хранилище за ключове, а публичният ключ е публикуван, манипулациите е невъзможно да се скрият.

Опитайте сами: променете `tool_name` или `agent_id` или `timestamp` в клетката по-горе и изпълнете отново. Всяка промяна произвежда невалидна разписка.


## Раздел 3: Свързване на разписки в низ

Една разписка защитава едно действие. Повечето агенти извършват много действия. За да направим цялата последователност видимо подправена, свързваме всяка разписка с предходната, като включваме хеша на предишната разписка в полезния товар на новата разписка.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ако някой премахне или пренареди разписка, низът се прекъсва точно в тази точка. Проверката на всяка по-късна разписка се проваля, тъй като нейният `previous_receipt_hash` вече не съвпада с действителния хеш на предшественика ѝ.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Сега прекъснете веригата, като фалшифицирате средното разписка и проверите отново. Фалшифицираната разписка не преминава проверката на подписа си, И следващата разписка не преминава проверката на връзката в веригата (тъй като `previous_receipt_hash` вече не съвпада с хеша на модифицираната средна разписка).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Квитанция 0 все още се проверява успешно (тя не е била променена и няма предшественик, от който да зависи). Квитанция 1 не преминава проверката на подписа си, защото променихме `tool_args_hash`. Квитанция 2 не преминава проверката на верижната връзка, защото нейният `previous_receipt_hash` е изчислен спрямо оригиналната (сега променена) квитанция 1.

Дори ако нападателят повторно подпише променената квитанция 1 (което не може да направи без частния ключ), несъответствието на верижната връзка в квитанция 2 все пак ще разкрие манипулацията. За да скрие промяната, нападателят би трябвало да подпише повторно всяка квитанция от точката на промяната нататък, което изисква притежание на частния ключ.


## Част 4: Обвийте повикване на агентски инструмент с подписване на разписка

При реално внедряване не искате всеки автор на агент да си спомня да извиква `make_receipt`. Искате подписването на разписки да бъде автоматично при всяко извикване на инструмент.

Ето най-простия шаблон: обвиващ клас, който приема всяка извикваема функция на инструмент и връща версия, която издава разписка. Това се адаптира към всяка агентска рамка, включително Microsoft Agent Framework (`agent_framework.foundry`).

Ако нямате създаден Microsoft Foundry проект, локалният макет по-долу все пак демонстрира шаблона.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Интегриране с Microsoft Agent Framework

Обвивката `ReceiptedTool` по-горе е независима от рамката. За да я използвате в агент, изграден с Microsoft Agent Framework, регистрирайте обвитата функция като инструмент. Скица (ще замените мока с реална регистрация на инструмент в Microsoft Foundry):

```python
# Псевдокод, показващ формата на интеграция.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Вие сте агент на Contoso Travel ...",
#     tools=[receipted_lookup],   # опакованият инструмент, не суровата функция
# )
# response = agent.run("Намерете полети от Сидни до Лос Анджелис през юни.")
#
# # След изпълнението, всяко извикване на инструмент от агента има подписана разписка:
# audit_chain = receipted_lookup.receipts
```

Агентската рамка не трябва да знае нищо за бележките за покупка. Подписването на бележки е обвито около инструмента, а не вградено в рамката. Така добавяте произход към съществуващ код на агент без да пренаписвате агента.


## Обобщение и предизвикателство за разширяване

Вие сте:

- Генерирали сте ключова двойка Ed25519.
- Създавали и подписвали сте разписка за извикване на агентски инструмент.
- Проверявали сте разписката офлайн, използвайки само публичния ключ.
- Подправяли сте разписка и сте наблюдавали провал на проверката.
- Създавали сте последователност от три разписки, свързани с хеш.
- Подправяли сте средата на веригата и сте наблюдавали както провал на подписа, така и провал на връзката в веригата.
- Обвивали сте функция на инструмент с автоматично подписване на разписки.

**Предизвикателство за разширяване.** Разширете схемата на разписката с поле `request_id` (UUID за разпределено проследяване). Актуализирайте `make_receipt`, за да го включва и потвърдете, че разписките все още се проверяват от край до край. След това модифицирайте полето след подписване и потвърдете, че проверката се проваля. Това ви принуждава да осмислите как всеки байт от каноничното кодиране допринася за подписа.

**Важна граница.** Разписките доказват три неща и само три неща: атрибуция (този ключ е подписал това съдържание), цялост (съдържанието не се е променяло след подписването) и подреждане (тази разписка е дошла след онази разписка). Те НЕ доказват, че действието на агента е било коректно, че политиката, посочена в `policy_id`, действително е била оценена или че агентът е спазвал всяко правило. Разписките са основа. Управлението е системата, която изграждате върху тях.

Прочетете отново README на урока с тази граница в ума си. Най-честата грешка, която екипите допускат с разписките, е да предполагат, че „имаме разписки“ означава „ние сме управлявани“. Не означава. Разписките правят поведението на агента проверимо. Те не го правят коректно.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
